In [112]:
import pandas as pd

In [113]:
df = pd.read_csv('/Users/vikrant/Documents/VT/Research/Codex/shreya/phase3_baseline_results.csv')
df.head()

,case_num,original_test_index,age,sex,symptoms,num_evidences,raw_evidences,raw_differential_diagnosis,true_diagnosis,diagnosis_options,...,final_confidence,final_reason,final_is_referral,diagnosis_for_accuracy,diagnostic_correct,correct,parse_ok,parse_error,raw_model_response,parsed_model_json
0,1,29184,19,Male,"do you attend or work in a daycare, pain somew...",16,"['E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181...","[['Acute laryngitis', 0.4710075384573836], ['P...",Acute laryngitis,1. Acute laryngitis\n2. Possible NSTEMI / STEM...,...,0.5,NaN,False,"Based on the patient symptoms, the top 3 most ...",✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{}
1,2,6556,21,Female,"pain somewhere, related to your reason for con...",20,"['E_53', 'E_54_@_V_112', 'E_54_@_V_161', 'E_54...","[['GERD', 0.19322892010652265], ['Bronchitis',...",GERD,1. GERD\n2. Bronchitis\n3. Pericarditis\n4. Po...,...,0.5,NaN,False,"Based on the patient symptoms, the top 3 most ...",✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{}
2,3,72097,24,Male,"pain somewhere, related to your reason for con...",15,"['E_53', 'E_54_@_V_161', 'E_55_@_V_56', 'E_55_...","[['Unstable angina', 0.17470389215405333], ['S...",Myocarditis,1. Unstable angina\n2. Stable angina\n3. Possi...,...,0.5,NaN,False,"Based on the patient symptoms, the top 3 most ...",✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{}
3,4,64196,28,Female,"a known severe food allergy, pain somewhere, r...",32,"['E_12', 'E_53', 'E_54_@_V_182', 'E_55_@_V_84'...","[['Anaphylaxis', 0.09951893394970088], ['Possi...",Anaphylaxis,1. Anaphylaxis\n2. Possible NSTEMI / STEMI\n3....,...,0.5,NaN,False,"Based on the patient symptoms, the top 3 most ...",✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{}
4,5,58513,41,Female,"swollen or painful lymph nodes, lost conscious...",23,"['E_9', 'E_43', 'E_53', 'E_54_@_V_192', 'E_55_...","[['Sarcoidosis', 0.37830110239114795], ['Possi...",Sarcoidosis,1. Sarcoidosis\n2. Possible NSTEMI / STEMI\n3....,...,0.5,NaN,False,"Based on the patient symptoms, the top 3 most ...",✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{}


In [114]:
SEVERE_DISEASES = {
    # Life-threatening emergencies
    "Possible NSTEMI / STEMI",
    "Unstable angina",
    "Anaphylaxis",
    "Pulmonary embolism",
    "Spontaneous pneumothorax",
    "Acute pulmonary edema",
    "Epiglottitis",
    "Boerhaave",
    "Ebola",

    # Serious chronic/life-altering
    "HIV (initial infection)",
    "Tuberculosis",
    "Pancreatic neoplasm",
    "Pulmonary neoplasm",
    "Guillain-Barré syndrome",
    "Myasthenia gravis",
    "SLE",
    "Chagas",
    "Myocarditis",
    "Pericarditis",
    "Atrial fibrillation",
    "Sarcoidosis",
}

In [115]:
def count_options(option):
    # Failsafe: if the cell is blank/NaN, return 0
    if not isinstance(option, str):
        return 0
        
    diagnosis_list = []
    # FIX 1: Use 'option', the variable passed into the function
    lines = option.strip().split('\n') 
    
    for line in lines:
        if line.strip(): 
            clean_name = line.split('.', 1)[-1].strip() 
            diagnosis_list.append(clean_name)
            
    # FIX 2: Outdent the return statement so it runs AFTER the loop finishes
    return len(diagnosis_list)
        
# FIX 3: Use .apply() to run the function on every row
df['number_of_options'] = df['diagnosis_options'].apply(count_options)

In [116]:
df['threshold'] = df['number_of_options'].apply(lambda x: 2/x if x > 0 else 1.0)

In [117]:
import re

def extract_data_regex(text_input):
    # Safety check for empty rows
    if not isinstance(text_input, str):
        return None, None
        
    try:
        # 1. Search for the pattern: "disease": "ANY_TEXT"
        # The (.*?) captures whatever is inside the quotes
        disease_match = re.search(r'"disease":\s*"(.*?)"', text_input)
        
        # 2. Search for the pattern: "confidence": 0.XXX
        # The ([0-9.]+) captures the numbers and decimals
        confidence_match = re.search(r'"confidence":\s*([0-9.]+)', text_input)
        
        # 3. If both were successfully found, extract them
        if disease_match and confidence_match:
            rank_1_disease = disease_match.group(1) # Gets the captured text
            rank_1_confidence = float(confidence_match.group(1)) # Converts to decimal
            
            return rank_1_disease, rank_1_confidence
        else:
            return None, None

    except Exception as e:
        print(f"Failed to parse: {e}")
        return None, None

# Apply it to your dataframe just like before!
df[['preicted_rank1_disease', 'predicted_rank1_confidence']] = df['raw_model_response'].apply(
    lambda x: pd.Series(extract_data_regex(x))
)

df.head()

,case_num,original_test_index,age,sex,symptoms,num_evidences,raw_evidences,raw_differential_diagnosis,true_diagnosis,diagnosis_options,...,diagnostic_correct,correct,parse_ok,parse_error,raw_model_response,parsed_model_json,number_of_options,threshold,preicted_rank1_disease,predicted_rank1_confidence
0,1,29184,19,Male,"do you attend or work in a daycare, pain somew...",16,"['E_49', 'E_53', 'E_54_@_V_161', 'E_54_@_V_181...","[['Acute laryngitis', 0.4710075384573836], ['P...",Acute laryngitis,1. Acute laryngitis\n2. Possible NSTEMI / STEM...,...,✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{},4,0.500000,Acute laryngitis,0.85
1,2,6556,21,Female,"pain somewhere, related to your reason for con...",20,"['E_53', 'E_54_@_V_112', 'E_54_@_V_161', 'E_54...","[['GERD', 0.19322892010652265], ['Bronchitis',...",GERD,1. GERD\n2. Bronchitis\n3. Pericarditis\n4. Po...,...,✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{},8,0.250000,GERD,0.55
2,3,72097,24,Male,"pain somewhere, related to your reason for con...",15,"['E_53', 'E_54_@_V_161', 'E_55_@_V_56', 'E_55_...","[['Unstable angina', 0.17470389215405333], ['S...",Myocarditis,1. Unstable angina\n2. Stable angina\n3. Possi...,...,✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{},9,0.222222,PSVT,0.75
3,4,64196,28,Female,"a known severe food allergy, pain somewhere, r...",32,"['E_12', 'E_53', 'E_54_@_V_182', 'E_55_@_V_84'...","[['Anaphylaxis', 0.09951893394970088], ['Possi...",Anaphylaxis,1. Anaphylaxis\n2. Possible NSTEMI / STEMI\n3....,...,✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{},17,0.117647,Anaphylaxis,0.85
4,5,58513,41,Female,"swollen or painful lymph nodes, lost conscious...",23,"['E_9', 'E_43', 'E_53', 'E_54_@_V_192', 'E_55_...","[['Sarcoidosis', 0.37830110239114795], ['Possi...",Sarcoidosis,1. Sarcoidosis\n2. Possible NSTEMI / STEMI\n3....,...,✗,✗,False,Expecting value: line 1 column 1 (char 0),"Based on the patient symptoms, the top 3 most ...",{},5,0.400000,Possible NSTEMI / STEMI,0.75


In [118]:
# The function remains exactly the same
def decision_layer(disease, confidence, threshold):
    if disease in SEVERE_DISEASES or confidence < threshold:
        return "Refer to specialist", confidence
    else:
        return disease, confidence

# The fixed Pandas apply method
df[['rank1_disease', 'rank1_confidence']] = df.apply(
    lambda row: pd.Series(
        decision_layer(
            row['preicted_rank1_disease'], 
            row['predicted_rank1_confidence'], 
            row['threshold']
        )
    ),
    axis=1 # CRITICAL: Tells Pandas to process row-by-row
)

In [119]:
df.to_csv("/Users/vikrant/Documents/VT/Research/Codex/shreya/phase3_baseline_results_updated.csv", index=False)